## Example 1: Basic argparse with argcomplete

In [ ]:
import argparse
import argcomplete
import sys

# Example 1: Basic Argument Parser with Auto-completion Setup
print("Example 1: Basic argparse with argcomplete")

try:
    # Create argument parser
    parser = argparse.ArgumentParser(description='Demo script with tab completion')
    
    # Add various argument types
    parser.add_argument('--verbose', '-v', action='store_true',
                       help='Enable verbose output')
    parser.add_argument('--output', '-o', type=str, 
                       help='Output file path')
    parser.add_argument('--format', choices=['json', 'xml', 'csv', 'yaml'],
                       help='Output format')
    parser.add_argument('--count', type=int, default=10,
                       help='Number of items to process')
    parser.add_argument('files', nargs='*', help='Input files')
    
    # Enable argcomplete (this enables tab completion)
    argcomplete.autocomplete(parser)
    
    print("✓ Argument parser created with argcomplete support")
    
    # Test parsing with sample arguments
    test_args = ['--verbose', '--format', 'json', '--count', '5', 'file1.txt']
    args = parser.parse_args(test_args)
    
    print(f"✓ Parsed arguments successfully:")
    print(f"  Verbose: {args.verbose}")
    print(f"  Format: {args.format}")
    print(f"  Count: {args.count}")
    print(f"  Files: {args.files}")
    
    assert (args.verbose, args.format, args.count, args.files) == (True, 'json', 5, ['file1.txt']), args
    
except Exception as e:
    print(f"Error in basic completion: {e}")
    raise AssertionError(
        f"argcomplete example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

## Example 2: Custom Completers for Dynamic Options


In [ ]:
import argparse
import argcomplete

# Example 2: Custom Completers for Dynamic Values
print("Example 2: Custom completion functions")

try:
    # Define custom completer functions
    def database_completer(prefix, parsed_args, **kwargs):
        """Custom completer that returns database names"""
        databases = ['users_db', 'products_db', 'analytics_db', 'logs_db']
        return [db for db in databases if db.startswith(prefix)]
    
    def server_completer(prefix, parsed_args, **kwargs):
        """Custom completer for server environments"""
        servers = ['dev-server', 'staging-server', 'prod-server', 'test-server']
        return [srv for srv in servers if srv.startswith(prefix)]
    
    # Create parser with custom completers
    parser = argparse.ArgumentParser(description='Database management tool')
    
    # Add arguments with custom completers
    parser.add_argument('--database', help='Database name').completer = database_completer
    parser.add_argument('--server', help='Server environment').completer = server_completer
    parser.add_argument('--action', choices=['backup', 'restore', 'migrate', 'status'],
                       help='Action to perform')
    
    # Enable argcomplete
    argcomplete.autocomplete(parser)
    
    print("✓ Custom completers configured")
    print("✓ Database completer provides: users_db, products_db, analytics_db, logs_db")
    print("✓ Server completer provides: dev-server, staging-server, prod-server, test-server")
    
    # Test completer functions directly
    db_suggestions = database_completer('user', None)
    server_suggestions = server_completer('dev', None)
    
    print(f"✓ Database suggestions for 'user': {db_suggestions}")
    print(f"✓ Server suggestions for 'dev': {server_suggestions}")
    
    # Test parsing
    test_args = ['--database', 'users_db', '--server', 'dev-server', '--action', 'backup']
    args = parser.parse_args(test_args)
    print(f"✓ Parsed: DB={args.database}, Server={args.server}, Action={args.action}")
    assert db_suggestions == ['users_db'] and server_suggestions == ['dev-server'], (db_suggestions, server_suggestions)
    
except Exception as e:
    print(f"Error in custom completers: {e}")
    raise AssertionError(
        f"argcomplete example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

## Example 3: File Path Completion


In [ ]:
import argparse
import os
import tempfile

import argcomplete
from argcomplete.completers import FilesCompleter, DirectoriesCompleter

# Example 3: File and Directory Path Completion
print("Example 3: File and directory completion")

try:
    parser = argparse.ArgumentParser(description='File processing tool')
    
    # File completion for input files
    parser.add_argument('--input-file', help='Input file path').completer = FilesCompleter()
    
    # Directory completion for output directory
    parser.add_argument('--output-dir', help='Output directory').completer = DirectoriesCompleter()
    
    # Specific file type completion (Python files)
    parser.add_argument('--script', help='Python script').completer = FilesCompleter(allowednames='*.py')
    
    # Configuration file completion (JSON/YAML)
    parser.add_argument('--config', help='Config file').completer = FilesCompleter(
        allowednames=['*.json', '*.yaml', '*.yml', '*.cfg', '*.ini']
    )
    
    # Log directory completion
    parser.add_argument('--log-dir', help='Log directory').completer = DirectoriesCompleter()
    
    # Enable argcomplete
    argcomplete.autocomplete(parser)
    
    print("✓ File and directory completers configured")
    print("✓ Input file: completes any file")
    print("✓ Output directory: completes directories only")
    print("✓ Script: completes *.py files only")
    print("✓ Config: completes *.json, *.yaml, *.yml, *.cfg, *.ini files")
    print("✓ Log directory: completes directories only")
    
    # Test with sample paths (would work with real filesystem)
    test_args = ['--input-file', '/tmp/test.txt', '--output-dir', '/tmp', '--script', 'main.py']
    args = parser.parse_args(test_args)
    
    print(f"✓ Parsed file arguments:")
    print(f"  Input file: {args.input_file}")
    print(f"  Output dir: {args.output_dir}")
    print(f"  Script: {args.script}")
    
    
    # Call the completers directly on a scratch directory
    with tempfile.TemporaryDirectory() as tmp:
        for name in ("main.py", "notes.txt"):
            open(os.path.join(tmp, name), "w").close()
        os.mkdir(os.path.join(tmp, "out"))
        prefix = os.path.join(tmp, "")
        py_matches = list(FilesCompleter(allowednames='*.py')(prefix=prefix))
        dir_matches = list(DirectoriesCompleter()(prefix=prefix))
    print(f"✓ *.py completer offers: {[os.path.relpath(m, tmp) for m in py_matches]}")
    print(f"✓ Directory completer offers: {[os.path.relpath(m, tmp) for m in dir_matches]}")
    assert os.path.join(tmp, "main.py") in py_matches and os.path.join(tmp, "notes.txt") not in py_matches, py_matches
    assert dir_matches == [os.path.join(tmp, "out", "")], dir_matches
    
except Exception as e:
    print(f"Error in file completion: {e}")
    raise AssertionError(
        f"argcomplete example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

## Example 4: Subparser Completion


In [ ]:
import argparse
import argcomplete

# Example 4: Subcommands with Completion
print("Example 4: Subcommand completion")

try:
    # Create main parser
    parser = argparse.ArgumentParser(description='Multi-command tool')
    parser.add_argument('--verbose', action='store_true', help='Verbose output')
    
    # Create subparsers for different commands
    subparsers = parser.add_subparsers(dest='command', help='Available commands')
    
    # User management subcommand
    user_parser = subparsers.add_parser('user', help='User management')
    user_parser.add_argument('action', choices=['create', 'delete', 'list', 'update'])
    user_parser.add_argument('--name', help='User name')
    user_parser.add_argument('--email', help='User email')
    user_parser.add_argument('--role', choices=['admin', 'user', 'guest'])
    
    # Database subcommand
    db_parser = subparsers.add_parser('database', help='Database operations')
    db_parser.add_argument('action', choices=['backup', 'restore', 'migrate', 'status'])
    db_parser.add_argument('--target', help='Target database')
    db_parser.add_argument('--format', choices=['sql', 'json', 'csv'])
    
    # Config subcommand
    config_parser = subparsers.add_parser('config', help='Configuration management')
    config_parser.add_argument('action', choices=['get', 'set', 'list', 'reset'])
    config_parser.add_argument('--key', help='Configuration key')
    config_parser.add_argument('--value', help='Configuration value')
    
    # Enable argcomplete
    argcomplete.autocomplete(parser)
    
    print("✓ Subparser completion configured")
    print("✓ Available commands: user, database, config")
    print("✓ Each subcommand has its own completion options")
    
    # Test subcommand parsing
    test_cases = [
        ['--verbose', 'user', 'create', '--name', 'john', '--role', 'admin'],
        ['database', 'backup', '--target', 'prod', '--format', 'sql'],
        ['config', 'set', '--key', 'timeout', '--value', '30']
    ]
    
    for i, test_args in enumerate(test_cases, 1):
        args = parser.parse_args(test_args)
        print(f"✓ Test case {i}: Command={args.command}")
        if hasattr(args, 'action'):
            print(f"  Action: {args.action}")
    
    commands = [parser.parse_args(t).command for t in test_cases]
    assert commands == ['user', 'database', 'config'], commands
    
except Exception as e:
    print(f"Error in subparser completion: {e}")
    raise AssertionError(
        f"argcomplete example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed\n")

## Example 5: Environment-aware completion

In [ ]:
import argparse
import argcomplete
import os

# Example 5: Environment-aware Completion
print("Example 5: Environment-based completion")

try:
    # Custom completer that uses environment variables
    def env_var_completer(prefix, parsed_args, **kwargs):
        """Complete with environment variable names"""
        env_vars = [key for key in os.environ.keys() if key.startswith(prefix.upper())]
        return env_vars[:10]  # Limit to 10 suggestions
    
    def path_completer(prefix, parsed_args, **kwargs):
        """Complete with PATH directories"""
        path_dirs = os.environ.get('PATH', '').split(os.pathsep)
        matching_dirs = [d for d in path_dirs if d.startswith(prefix)]
        return matching_dirs[:5]  # Limit to 5 suggestions
    
    # Create parser with environment-aware completion
    parser = argparse.ArgumentParser(description='Environment-aware tool')
    
    # Environment variable completion
    parser.add_argument('--env-var', help='Environment variable').completer = env_var_completer
    
    # PATH directory completion
    parser.add_argument('--path-dir', help='Directory from PATH').completer = path_completer
    
    # Python path completion
    parser.add_argument('--python-module', help='Python module')
    
    # Working directory context
    parser.add_argument('--working-dir', default=os.getcwd(), help='Working directory')
    
    # Enable argcomplete
    argcomplete.autocomplete(parser)
    
    print("✓ Environment-aware completers configured")
    
    # Test environment variable completion
    env_suggestions = env_var_completer('PATH', None)
    print(f"✓ Environment variables starting with 'PATH': {env_suggestions[:3]}")
    
    # Test PATH completion
    path_suggestions = path_completer('/usr', None)
    print(f"✓ PATH directories starting with '/usr': {path_suggestions[:3]}")
    
    # Show current environment context
    print(f"✓ Current working directory: {os.getcwd()}")
    print(f"✓ Python path: {os.environ.get('PYTHONPATH', 'Not set')}")
    
    # Test parsing with environment values
    test_args = ['--env-var', 'PATH', '--working-dir', '/tmp']
    args = parser.parse_args(test_args)
    
    print(f"✓ Parsed environment-aware arguments:")
    print(f"  Environment variable: {args.env_var}")
    print(f"  Working directory: {args.working_dir}")
    
    assert all(name.startswith('PATH') for name in env_suggestions), env_suggestions
    assert args.env_var == 'PATH' and args.working_dir == '/tmp', args
    
except Exception as e:
    print(f"Error in environment completion: {e}")
    raise AssertionError(
        f"argcomplete example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed")